# OmniPhyto: molecular dynamics on Kaggle (runs in the background)

Same simulation as the Colab notebook (OpenMM, candidate vs the crystal drug 605), but it runs **on Kaggle's servers**: you start it, close the laptop, and come back later.

## One-time setup (about 10 minutes)
1. **Make a Kaggle account** and verify your phone number (needed for GPU and Internet).
2. **Create a dataset** (Datasets > New Dataset) called `omniphyto-md-inputs`, and upload these files from your Drive `Herbaquest/omniphyto_run/`:
   - `receptors/dpp4_H.pdb`
   - `receptors/ligand_605_crystal.pdb`
   - `results/target/poses/OP0145_out.pdbqt`, `OP0048_out.pdbqt` and `OP0437_out.pdbqt` (the three default compounds). If you add `OP0385` or `OP0019` in the settings, upload their `_out.pdbqt` files too
3. **Open this notebook on Kaggle** (File > Import Notebook), then in the right-hand panel:
   - **Add Input** > your dataset `omniphyto-md-inputs`
   - **Accelerator**: GPU T4 x2 (or P100)
   - **Internet**: On
4. Check the settings in the next cell, then press **Save Version > Save & Run All (Commit)**. It runs in the background for up to about 11 hours. You can close the page.

## When it finishes
Open the notebook's **Output** tab and download `md_report` (plots and `md_summary.csv`). Put that folder in Drive as `Herbaquest/omniphyto_run/md_report/`, then run the Drive-to-GitHub notebook.

If it stops at the time limit, add this notebook's previous output as an Input (Add Input > Notebook Output) and run it again. It continues from the saved pieces.

In [ ]:
# CELL 1: settings and a check that your input files are there
import os, glob, time
from pathlib import Path
T0 = time.time()

# ===================== SETTINGS =====================
CANDIDATES    = ['OP0145', 'OP0048', 'OP0437']   # OP0145 = rank 1 overall; OP0048 = best docking score (-9.39) and efficiency 0.35; OP0437 = weak binder (-4.2), the NEGATIVE control
# Optional extras (each adds about 3 hours of GPU time and needs its pose file in your dataset):
#   'OP0385'  rank 3 overall, strong contacts with Tyr547/Ser630/Arg125     'OP0019'  most efficient small compound (24 heavy atoms, efficiency 0.37)
CONTROL       = True            # also simulate the crystal drug 605
PRODUCTION_NS = 10              # ns per system
BUDGET_H      = 10.5            # stop starting new pieces after this many hours (Kaggle sessions end at about 12 h)
PADDING_NM    = 1.0
# ====================================================

DATA = os.environ.get('OMNI_DATA', '/kaggle/input')
OUT  = Path(os.environ.get('OMNI_OUT', '/kaggle/working'))
WORK, REPORT = OUT / 'md_work', OUT / 'md_report'
NAMES = (['605'] if CONTROL else []) + list(CANDIDATES)
need = ['dpp4_H.pdb'] + (['ligand_605_crystal.pdb'] if CONTROL else []) + [f'{c}_out.pdbqt' for c in CANDIDATES]
missing = [n for n in need if not glob.glob(f'{DATA}/**/{n}', recursive=True)]
for n in need: print(('OK       ' if n not in missing else 'MISSING  ') + n)
if missing: raise SystemExit('Add the missing files to your Kaggle dataset (see the first cell), then run again.')
print('All input files found. Systems to run:', NAMES)

In [ ]:
# CELL 2: install OpenMM and the ligand tools in a private environment (about 5-10 minutes)
import subprocess, shutil, re, os
from pathlib import Path
ENV = Path(os.environ.get('OMNI_ENV', '/tmp/mmenv')); PY = ENV / 'bin' / 'python'
RUNENV = {**os.environ, 'PATH': f'{ENV}/bin:' + os.environ['PATH'], 'MPLBACKEND': 'Agg'}
if not PY.exists():
    mmb = Path('/tmp/micromamba')
    if not mmb.exists():
        subprocess.run(['curl', '-sL', '-o', str(mmb), 'https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64'], check=True); mmb.chmod(0o755)
    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
    drv = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', smi)
    if not drv: raise SystemExit('No GPU found. In the right-hand panel set Accelerator to GPU (T4 x2 or P100).')
    print('GPU driver supports', drv.group(0))
    pin = f'cuda-version>=12,<={drv.group(1)}.{drv.group(2)}'
    r = subprocess.run([str(mmb), 'create', '-y', '-q', '-p', str(ENV), '-c', 'conda-forge', 'python=3.11', 'openmm', 'openmmforcefields', 'openff-toolkit', 'ambertools', 'mdtraj', 'rdkit', 'pdbfixer', 'matplotlib', 'pandas', pin],
                       capture_output=True, text=True, env={**os.environ, 'MAMBA_ROOT_PREFIX': '/tmp/mamba'})
    if r.returncode: raise SystemExit('Install failed:\n' + r.stdout[-1500:] + r.stderr[-1500:])
    subprocess.run([str(PY), '-m', 'pip', 'install', '-q', 'meeko'], check=True)
test = subprocess.run([str(PY), '-c', "import openmm as m\nprint('OpenMM', m.__version__, [m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\ns=m.System(); s.addParticle(1.0)\ntry:\n    c=m.Context(s, m.VerletIntegrator(0.001), m.Platform.getPlatformByName('CUDA')); print('CUDA works')\nexcept Exception as e:\n    print('CUDA FAILED:', e)"], capture_output=True, text=True, env=RUNENV)
print(test.stdout, test.stderr[-500:])
if 'CUDA works' not in test.stdout and not os.environ.get('OMNI_ENV'): raise SystemExit('The GPU could not be used by OpenMM. Send me this output.')

In [ ]:
# CELL 3: write the simulation script, and pick up saved pieces from an earlier run if you added one as an Input
SCRIPT = r'''# OmniPhyto MD runner (OpenMM). Stages: run (build + equilibrate + produce, resumable) and analyze.
import os, sys, re, glob, time, shutil, argparse
os.environ['MPLBACKEND'] = 'Agg'      # Kaggle sets a notebook-only plotting backend that this environment does not have
from pathlib import Path
import numpy as np, pandas as pd

ap = argparse.ArgumentParser()
ap.add_argument('stage', choices=['run', 'analyze'])
ap.add_argument('--data', required=True)          # folder searched recursively for the input files
ap.add_argument('--work', required=True)
ap.add_argument('--report', required=True)
ap.add_argument('--names', nargs='+', required=True)
ap.add_argument('--control', default='605')
ap.add_argument('--ns', type=float, default=10.0)
ap.add_argument('--deadline', type=float, default=0.0)   # absolute time (epoch seconds) after which no new piece is started
ap.add_argument('--device', default='0')
ap.add_argument('--padding', type=float, default=1.0)
ap.add_argument('--chunk-ns', type=float, default=0.5)
ap.add_argument('--dcd-steps', type=int, default=5000)
ap.add_argument('--equil-scale', type=float, default=1.0)
ap.add_argument('--seed', type=int, default=2026)
A = ap.parse_args()

WORK = Path(A.work); WORK.mkdir(parents=True, exist_ok=True)
REPORT = Path(A.report); REPORT.mkdir(parents=True, exist_ok=True)
SMILES_CONTROL = '[NH3+][C@H](Cc1cc(F)ccc1F)c1ccc(cc1)-c1cccc(c1)C(N)=O'      # crystal drug 605, charged as at pH 7.4
TEMP = 300

def find(name):
    hits = sorted(glob.glob(f'{A.data}/**/{name}', recursive=True))
    if not hits: raise SystemExit(f'Input file not found: {name} (looked under {A.data})')
    return Path(hits[0])

def label(n): return 'control' if n == A.control else 'candidate'

# ============================== analysis ==============================
def analyze():
    import mdtraj as md, matplotlib
    matplotlib.use('Agg'); import matplotlib.pyplot as plt
    rows = []
    for name in A.names:
        d = WORK / name
        parts = sorted(d.glob('prod_*.dcd'))
        if not parts: print(name, 'no trajectory yet'); continue
        top_pdb = str(d / 'equil.pdb')
        t = md.join([md.load(str(p), top=top_pdb) for p in parts]); t.image_molecules(inplace=True)
        top = t.topology
        LA = set(int(i) for i in top.select('resname LIG'))
        prot_bb = np.array([i for i in top.select('protein and backbone') if int(i) not in LA])
        ca = np.array([i for i in top.select('protein and name CA') if int(i) not in LA])
        lig = [i for i in sorted(LA) if top.atom(i).element.symbol != 'H']
        ns = np.arange(t.n_frames) * (A.dcd_steps * 0.004 / 1000)
        t.superpose(t, 0, atom_indices=prot_bb)
        rmsd_prot = md.rmsd(t, t, 0, atom_indices=prot_bb, superpose=False) * 10
        dl = t.xyz[:, lig, :] - t.xyz[0, lig, :][None]
        rmsd_lig = np.sqrt((dl ** 2).sum(2).mean(1)) * 10
        k = max(1, int(0.1 * t.n_frames))
        tt = t[k:]; tt.superpose(tt, 0, atom_indices=prot_bb)
        rmsf = md.rmsf(tt, tt, 0, atom_indices=ca) * 10
        resid = [top.atom(i).residue.resSeq for i in ca]
        hb = md.baker_hubbard(t, freq=0.0, exclude_water=True, periodic=False)
        cand = np.array([x for x in hb if (int(x[0]) in LA) != (int(x[2]) in LA)], dtype=int).reshape(-1, 3)
        frames_hb = np.zeros(t.n_frames, bool); per_bond = []
        if len(cand):
            dist = md.compute_distances(t, cand[:, [1, 2]], periodic=False)
            ang = md.compute_angles(t, cand, periodic=False)
            ok = (dist < 0.25) & (ang > np.deg2rad(120))
            frames_hb = ok.any(1)
            for j, (a, b, c) in enumerate(cand):
                per_bond.append(dict(donor=str(top.atom(a)), hydrogen=str(top.atom(b)), acceptor=str(top.atom(c)), occupancy_pct=round(100 * ok[:, j].mean(), 1)))
        if per_bond: pd.DataFrame(per_bond).sort_values('occupancy_pct', ascending=False).to_csv(REPORT / f'{name}_hbonds_detail.csv', index=False)
        hb_pct = 100 * frames_hb[k:].mean()
        prot_heavy = np.array([i for i in top.select('protein and not element H') if int(i) not in LA])
        nb = md.compute_neighbors(t[k:], 0.4, lig, haystack_indices=prot_heavy)
        cnt = {}
        for fr in nb:
            for r in {top.atom(i).residue for i in fr}: cnt[str(r)] = cnt.get(str(r), 0) + 1
        contacts = sorted(((v / len(nb), r) for r, v in cnt.items()), reverse=True)
        top_c = ', '.join(f'{r} {100*v:.0f}%' for v, r in contacts[:6])
        lab = label(name)
        fig, ax = plt.subplots(figsize=(6, 3.2)); ax.plot(ns, rmsd_prot, label='protein backbone'); ax.plot(ns, rmsd_lig, label='ligand (fitted on protein)')
        ax.set_xlabel('time (ns)'); ax.set_ylabel('RMSD (Å)'); ax.set_title(f'{name} ({lab}): RMSD'); ax.legend(); fig.tight_layout(); fig.savefig(REPORT / f'{name}_rmsd.png', dpi=150); plt.close(fig)
        fig, ax = plt.subplots(figsize=(6, 3.2)); ax.plot(resid, rmsf); ax.set_xlabel('residue number'); ax.set_ylabel('C-alpha RMSF (Å)'); ax.set_title(f'{name} ({lab}): RMSF'); fig.tight_layout(); fig.savefig(REPORT / f'{name}_rmsf.png', dpi=150); plt.close(fig)
        fig, ax = plt.subplots(figsize=(6, 3.2)); ax.plot(ns, np.convolve(frames_hb.astype(float), np.ones(10) / 10, mode='same') * 100)
        ax.set_xlabel('time (ns)'); ax.set_ylabel('% frames with an H-bond (smoothed)'); ax.set_ylim(-2, 102); ax.set_title(f'{name} ({lab}): ligand-protein H-bonds'); fig.tight_layout(); fig.savefig(REPORT / f'{name}_hbonds.png', dpi=150); plt.close(fig)
        rows.append(dict(compound_id=name, label=lab, length_ns=round(float(ns[-1] + A.dcd_steps * 0.004 / 1000), 2), ligand_rmsd_mean_A=round(float(rmsd_lig[k:].mean()), 2),
                         ligand_rmsd_sd_A=round(float(rmsd_lig[k:].std()), 2), protein_rmsd_mean_A=round(float(rmsd_prot[k:].mean()), 2),
                         hbond_occupancy_pct=round(float(hb_pct), 1), notes=f'main contacts: {top_c}'))
        print(name, rows[-1])
    if rows:
        df = pd.DataFrame(rows); df.to_csv(REPORT / 'md_summary.csv', index=False)
        print(df.to_string(index=False)); print('Saved to', REPORT)

if A.stage == 'analyze':
    analyze(); sys.exit(0)

# ============================== run ==============================
import openmm as mm, openmm.app as app, openmm.unit as u
from openmmforcefields.generators import GAFFTemplateGenerator
from openff.toolkit import Molecule
from rdkit import Chem
from rdkit.Chem import AllChem

PROT = find('dpp4_H.pdb')

def ligand_from_pose(cid):
    from meeko import PDBQTMolecule, RDKitMolCreate
    pm = PDBQTMolecule.from_file(str(find(f'{cid}_out.pdbqt')), skip_typing=True)
    m = Chem.AddHs(RDKitMolCreate.from_pdbqt_mol(pm)[0], addCoords=True)
    for a in m.GetAtoms(): a.SetMonomerInfo(None)
    return m

def ligand_from_crystal(code='605'):
    xt = Chem.MolFromPDBFile(str(find(f'ligand_{code}_crystal.pdb')), removeHs=True, sanitize=False)
    m = Chem.AddHs(AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(SMILES_CONTROL), xt), addCoords=True)
    for a in m.GetAtoms(): a.SetMonomerInfo(None)
    return m

def check_pose(name, mol):
    xyz = mol.GetConformer().GetPositions(); heavy = np.array([a.GetAtomicNum() > 1 for a in mol.GetAtoms()])
    pr = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in open(PROT) if l.startswith('ATOM') and l[76:78].strip() != 'H'])
    d = np.linalg.norm(xyz[heavy][:, None, :] - pr[None, :, :], axis=2)
    print(f'[{name}] {int(heavy.sum())} heavy atoms, closest protein heavy atom {d.min():.2f} A, {(d < 4.0).any(0).sum()} protein atoms within 4 A', flush=True)

def build_system(name):
    d = WORK / name; d.mkdir(exist_ok=True)
    if (d / 'system.xml').exists() and (d / 'start.pdb').exists(): return d
    mol = ligand_from_crystal(name) if name == A.control else ligand_from_pose(name)
    check_pose(name, mol)
    prot = app.PDBFile(str(PROT))
    off = Molecule.from_rdkit(mol, allow_undefined_stereo=True)
    gen = GAFFTemplateGenerator(molecules=off, forcefield='gaff-2.11')
    ff = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml'); ff.registerTemplateGenerator(gen.generator)
    lig_top = off.to_topology().to_openmm()
    for r in lig_top.residues(): r.name = 'LIG'
    assert lig_top.getNumResidues() == 1, 'the ligand was split into several residues'
    mod = app.Modeller(prot.topology, prot.positions)
    mod.add(lig_top, (mol.GetConformer().GetPositions() / 10.0) * u.nanometer)
    mod.addSolvent(ff, model='tip3p', padding=A.padding * u.nanometer, ionicStrength=0.15 * u.molar, neutralize=True)
    system = ff.createSystem(mod.topology, nonbondedMethod=app.PME, nonbondedCutoff=1.0 * u.nanometer, constraints=app.HBonds, hydrogenMass=1.5 * u.amu, rigidWater=True)
    for r in mod.topology.residues(): r.id = str(r.id)
    (d / 'system.xml').write_text(mm.XmlSerializer.serialize(system))
    with open(d / 'start.pdb', 'w') as fh: app.PDBFile.writeFile(mod.topology, mod.positions, fh, keepIds=True)
    n_lig = sum(1 for a in mod.topology.atoms() if a.residue.name == 'LIG')
    print(f'[{name}] built: {mod.topology.getNumAtoms()} atoms ({n_lig} ligand atoms)', flush=True)
    return d

def make_sim(d, k0=1000.0):
    system = mm.XmlSerializer.deserialize((d / 'system.xml').read_text())
    pdb = app.PDBFile(str(d / 'start.pdb'))
    f = mm.CustomExternalForce('k*periodicdistance(x,y,z,x0,y0,z0)^2')
    f.addGlobalParameter('k', k0)
    for p_ in ('x0', 'y0', 'z0'): f.addPerParticleParameter(p_)
    for a in pdb.topology.atoms():
        if a.residue.name not in ('HOH', 'WAT', 'NA', 'CL', 'Na+', 'Cl-') and a.element is not None and a.element.atomic_number > 1:
            f.addParticle(a.index, pdb.positions[a.index].value_in_unit(u.nanometer))
    system.addForce(f)
    system.addForce(mm.MonteCarloBarostat(1.0 * u.bar, TEMP * u.kelvin, 25))
    integ = mm.LangevinMiddleIntegrator(TEMP * u.kelvin, 1.0 / u.picosecond, 0.004 * u.picoseconds); integ.setRandomNumberSeed(A.seed)
    sim = None
    for pname, props in (('CUDA', {'Precision': 'mixed', 'DeviceIndex': str(A.device)}), ('OpenCL', {'Precision': 'mixed'}), ('CPU', {})):
        try:
            plat = mm.Platform.getPlatformByName(pname); sim = app.Simulation(pdb.topology, system, integ, plat, props); break
        except Exception as e:
            print(f'  platform {pname} not usable: {str(e)[:110]}', flush=True)
    if sim is None: raise SystemExit('No OpenMM platform could be created.')
    sim.context.setPositions(pdb.positions)
    return sim, plat.getName()

def equilibrate(name, d):
    if (d / 'equil.chk').exists(): return
    sim, plat = make_sim(d)
    if plat == 'CPU' and A.equil_scale >= 1: raise SystemExit('No GPU platform is working (it would take days on the CPU).')
    S = A.equil_scale
    print(f'[{name}] equilibration on {plat}', flush=True)
    sim.minimizeEnergy(maxIterations=max(50, int(5000 * S)))
    sim.context.setVelocitiesToTemperature(TEMP * u.kelvin, A.seed)
    sim.context.setParameter('k', 1000.0); sim.step(max(1, int(25000 * S)))
    for k in (1000.0, 300.0, 100.0): sim.context.setParameter('k', k); sim.step(max(1, int(16666 * S)))
    sim.context.setParameter('k', 0.0)
    sim.saveCheckpoint(str(d / 'equil.chk'))
    st = sim.context.getState(getPositions=True, enforcePeriodicBox=True)
    with open(d / 'equil.pdb', 'w') as fh: app.PDBFile.writeFile(sim.topology, st.getPositions(), fh, keepIds=True)

def produce(name, d):
    for tmp in d.glob('prod_*.dcd.tmp'): tmp.unlink()
    n_chunks = int(round(A.ns / A.chunk_ns)); steps = int(A.chunk_ns * 1000 / 0.004)
    k0 = len(sorted(d.glob('prod_*.dcd')))
    if k0 >= n_chunks: print(f'[{name}] finished ({k0 * A.chunk_ns:.2f} ns)', flush=True); return True
    sim, plat = make_sim(d)
    sim.loadCheckpoint(str(d / 'prod.chk' if (d / 'prod.chk').exists() else d / 'equil.chk')); sim.context.setParameter('k', 0.0)
    print(f'[{name}] production from {k0 * A.chunk_ns:.2f} ns on {plat}', flush=True)
    est = 5 * 60.0
    for k in range(k0, n_chunks):
        if A.deadline and time.time() + est * 1.3 + 120 > A.deadline:
            print(f'[{name}] time budget reached at {k * A.chunk_ns:.2f} of {A.ns} ns; run again to continue.', flush=True); return False
        part = d / f'prod_{k:03d}.dcd'
        rep = app.DCDReporter(str(part) + '.tmp', A.dcd_steps); sim.reporters = [rep]
        t = time.time(); sim.step(steps); sim.reporters = []; rep._out.close()
        os.replace(str(part) + '.tmp', part); sim.saveCheckpoint(str(d / 'prod.chk')); est = time.time() - t
        print(f'[{name}] {(k + 1) * A.chunk_ns:.2f} / {A.ns} ns  ({A.chunk_ns / (est / 86400):.0f} ns/day)', flush=True)
    return True

for name in A.names:
    d = build_system(name)
    if A.ns <= 0: continue
    equilibrate(name, d)
    if not produce(name, d): sys.exit(3)
print('RUN COMPLETE', A.names, flush=True)
'''
open('/tmp/omniphyto_md.py', 'w').write(SCRIPT)
import shutil
WORK.mkdir(parents=True, exist_ok=True)
for old in glob.glob(f'{DATA}/**/md_work', recursive=True):
    for sub in Path(old).iterdir():
        if sub.is_dir() and not (WORK / sub.name).exists():
            shutil.copytree(sub, WORK / sub.name); print('Continuing from earlier run:', sub.name)
print('script written')

In [ ]:
# CELL 4: run (the control and the candidate(s)); one system per GPU, in parallel when there are two GPUs
import subprocess, time, os
ngpu = max(1, len([l for l in subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.splitlines() if l.startswith('GPU')])) if not os.environ.get('OMNI_ONE_GPU') else 1
deadline = T0 + BUDGET_H * 3600
EXTRA = os.environ.get('OMNI_EXTRA', '').split()
LOGS = OUT / 'md_logs'; LOGS.mkdir(exist_ok=True)
pending = list(NAMES); running = {}; status = {}; seen = {}
import math
print(f'{ngpu} GPU(s); {len(NAMES)} system(s); budget {BUDGET_H} h')
print(f'Rough estimate at T4 speed: about {math.ceil(len(NAMES) / ngpu) * PRODUCTION_NS / 3.5:.0f} h in total ({PRODUCTION_NS} ns takes about 3 h per system). If this is more than the budget, the last system(s) stop early; shorten PRODUCTION_NS or remove a compound.')
def launch(name, dev):
    cmd = [str(PY), '/tmp/omniphyto_md.py', 'run', '--data', DATA, '--work', str(WORK), '--report', str(REPORT), '--names', name, '--ns', str(PRODUCTION_NS),
           '--deadline', str(deadline), '--device', str(dev), '--padding', str(PADDING_NM)] + EXTRA
    f = open(LOGS / f'{name}.log', 'w'); return subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, env=RUNENV), f
def show(name):
    lines = [l for l in open(LOGS / f'{name}.log').read().splitlines() if l.strip() and 'WARNING' not in l and 'dcdplugin' not in l]
    for l in lines[seen.get(name, 0):]: print(l)
    seen[name] = len(lines)
free = list(range(ngpu))
while pending or running:
    while pending and free:
        n = pending.pop(0); d = free.pop(0); running[n] = (d, *launch(n, d)); print(f'started {n} on GPU {d}')
    time.sleep(20)
    for n in list(running):
        d, p, f = running[n]; show(n)
        if p.poll() is not None:
            f.close(); show(n); status[n] = p.returncode; free.append(d); del running[n]
            print(f'{n}: ' + {0: 'finished', 3: 'stopped at the time budget (run again to continue)'}.get(p.returncode, f'FAILED (code {p.returncode}), see md_logs/{n}.log'))
print('status:', status)

In [ ]:
# CELL 5: analysis (RMSD, RMSF, H-bonds, md_summary.csv) of everything that has been simulated so far
r = subprocess.run([str(PY), '/tmp/omniphyto_md.py', 'analyze', '--data', DATA, '--work', str(WORK), '--report', str(REPORT), '--names', *NAMES] + EXTRA, capture_output=True, text=True, env=RUNENV)
print('\n'.join(l for l in (r.stdout + r.stderr).splitlines() if 'dcdplugin' not in l and 'WARNING' not in l)[-3000:])
print('\nFiles to download from the Output tab:', sorted(p.name for p in REPORT.glob('*')))